[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EstebanGZam/taller_agente/blob/main/secop_agent_colab.ipynb)

# Workshop 1 · SECOP procurement analyst agent

This notebook reproduces the complete Workshop 1 deliverable:

1. Clone the repository and install the package.
2. Load the keys from Colab secrets.
3. Offline checks (no language model, no cost).
4. One sample question to the Q&A agent.
5. Experiments: configuration A (baseline) vs. B (improved) in LangSmith, plus the comparison table.
6. Eligibility of the company in the 20 tender documents.
7. Web frontend (Gradio).
8. Where to look in LangSmith.

Workshop criterion: **at least 85% accuracy at the lowest possible cost**. The agents answer in Spanish
because the reference answers are in Spanish.

## 1. Clone the repository and install the package

In [ ]:
import os

REPO_URL = "https://github.com/EstebanGZam/taller_agente"
REPO_DIR = "/content/taller_agente"
if not os.path.exists(REPO_DIR):
    !git clone -q -b main {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -e ".[dev]"  # the secop_agent package plus pytest and ruff for the offline checks

## 2. Keys and configuration

Store `ANTHROPIC_API_KEY` and `LANGSMITH_API_KEY` in Colab **Secrets** (key icon in the left sidebar) and
grant this notebook access. Missing keys are requested with `getpass`. Keys are never printed.

Do not set `LANGSMITH_AGENT_ID` or `LANGSMITH_AGENT_ENVIRONMENT`: together with `LANGSMITH_PROJECT` they
stop LangSmith from recording traces, so they are removed here.

In [ ]:
import os
from getpass import getpass

for name in ("LANGSMITH_AGENT_ID", "LANGSMITH_AGENT_ENVIRONMENT"):
    os.environ.pop(name, None)


def read_secret(name: str) -> str:
    """Colab secret, or an interactive prompt when the secret is missing or not shared."""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:  # Outside Colab, or the secret does not exist / has no access.
        pass
    return getpass(f"{name}: ")


for key in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY"):
    if not os.environ.get(key):
        os.environ[key] = read_secret(key)

TEAM_NAME = "my-team"  # <- change to your team name
os.environ.update({
    "TEAM_NAME": TEAM_NAME,
    "LANGSMITH_TRACING": "true",
    "LANGSMITH_PROJECT": f"secop-taller1-{TEAM_NAME}",
    "MODEL": "anthropic:claude-haiku-5-5",
    "DATA_MODE": "sample",
    "AGENT_CONFIG": "B",
})
print("Keys loaded:", {key: bool(os.environ.get(key)) for key in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY")})
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

## 3. Offline checks

`secop_agent.procurement_data` runs the data tools in plain Python. `pytest` checks, without keys or
network, that the tools return the facts behind every reference question, that configuration A is the
course's original agent, and the 20 eligibility verdicts; `ruff` checks the code style.

In [ ]:
!python -m secop_agent.procurement_data
!python -m pytest -q && ruff check .

## 4. One question to the agent

`ask()` returns the answer and the tools the agent decided to call. `CostTracker` adds up tokens and cost
with Anthropic's public rates.

In [ ]:
from secop_agent.costs import CostTracker
from secop_agent.qa_agent import ask, normalize_config
from secop_agent.settings import model_from_env

tracker = CostTracker(model_from_env())
result = ask("¿Nuestro capital de trabajo alcanza el 40 % del presupuesto del proceso "
             "FLO-INFRA-SA-MC-007-2025?", callbacks=[tracker])
print(result.answer)
print("\nTools used:", ", ".join(result.tools_used) or "none")
print(f"Configuration {normalize_config()} · {model_from_env()} ·", tracker.summary())

## 5. Experiments: configuration A vs. B

- **A**: the course's original prompt and tool descriptions (baseline).
- **B**: precise tool descriptions, compact output with a summary and data-quality warnings, and a prompt
  with one rule per question type.

Each run evaluates the 18 reference questions in LangSmith (dataset `secop-taller1-referencia`) with two
evaluators (tool usage and answer correctness by an LLM judge) and saves
`results/qa_experiment_<prefix>.json` with accuracy, tokens, cost and latency.

In [ ]:
!AGENT_CONFIG=A python -m secop_agent.evaluate_qa A-baseline
!AGENT_CONFIG=B python -m secop_agent.evaluate_qa B-improved

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

!python -m secop_agent.compare_experiments A-baseline B-improved > /dev/null
display(Markdown(Path("results/qa_comparison.md").read_text(encoding="utf-8")))

## 6. Eligibility in the 20 tender documents

Reads the 20 PDFs, extracts the requirements, checks them against `empresa_multiproposito.json` and
compares each verdict (ELIGIBLE, NOT_ELIGIBLE, NOT_APPLICABLE) with the key in `data/eligibility_key.json`.
Name, reference and cost of every process are recorded in `results/eligibility_processes.{csv,md,json}`.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

!python -m secop_agent.batch_eligibility --company empresa_multiproposito.json
display(Markdown(Path("results/eligibility_processes.md").read_text(encoding="utf-8")))

In [ ]:
!python -m secop_agent.evaluate_eligibility eligibility-haiku

## 7. Frontend

Opens the Gradio interface with a temporary public link (`share=True`). Stop the cell to close it.

In [ ]:
from secop_agent.web_app import build_demo

build_demo().queue().launch(share=True, debug=True)

## 8. Where to look in LangSmith

At [smith.langchain.com](https://smith.langchain.com):

- **Projects → `secop-taller1-<team>`**: one trace per agent question (section 4 and the frontend), with the
  reason–act tree, the tools called, and the tokens and latency of each step.
- **Datasets & Experiments → `secop-taller1-referencia`**: the `A-baseline-…` and `B-improved-…`
  experiments. Each row shows the output (with `metrics`: tokens, cost and latency) and the feedback
  columns `tool_usage`, `correct_answer` and `cost_usd`.
- Select both experiments and click **Compare** to see answers, scores and the **Latency**, **Tokens** and
  **Cost** columns side by side. When LangSmith does not know the model's rate the Cost column is empty;
  the reference cost is ours (`cost_usd` column and `results/qa_comparison.md`).
- Each experiment's metadata holds `agent_config`, `model`, `judge_model`, `data_mode` and `team_name`;
  the traces carry the tags `config-A` / `config-B`.
- **Datasets & Experiments**: the `eligibility-haiku-…` experiment of the eligibility evaluation.